# 04 — Modeling Baseline (Exp 0 + Exp 1)
**Purpose:** Establish spatial CV framework + baselines.

**Exp 0:** Naïve baselines (mean predictor) — water quality only

**Exp 1:** Default XGBoost, RF, Ridge with Landsat + TerraClimate features

**Figures:**
- 📊 Spatial CV fold distribution map
- 📊 Exp 0 vs Exp 1 R² comparison bar chart
- 📊 Per-fold R² boxplots
- 📊 Model comparison heatmap (target × model)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.base import clone
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11, 'axes.titleweight': 'bold', 'figure.dpi': 120})

SEED = 42
WORK_DIR = '/kaggle/working'

In [ ]:
# ============================================================
# INLINE: Spatial CV (from src/spatial_cv.py)
# No need to import — self-contained notebook
# ============================================================
from sklearn.model_selection import BaseCrossValidator

class LeaveStationGroupOut(BaseCrossValidator):
    """Leave-Station-Group-Out CV. Each fold holds out entire stations."""
    def __init__(self, n_splits=10, station_col='GEMS Station Number', random_state=42):
        self.n_splits = n_splits
        self.station_col = station_col
        self.random_state = random_state
    
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits
    
    def split(self, X, y=None, groups=None):
        unique_stations = X[self.station_col].unique()
        rng = np.random.RandomState(self.random_state)
        rng.shuffle(unique_stations)
        station_folds = {s: i % self.n_splits for i, s in enumerate(unique_stations)}
        
        for fold_idx in range(self.n_splits):
            test_stations = [s for s, f in station_folds.items() if f == fold_idx]
            test_mask = X[self.station_col].isin(test_stations)
            yield X.index[~test_mask].values, X.index[test_mask].values
    
    def get_fold_station_mapping(self, X):
        mapping = {}
        for fold_idx, (_, test_idx) in enumerate(self.split(X)):
            mapping[fold_idx] = X.loc[test_idx, self.station_col].unique().tolist()
        return mapping

print('✅ Spatial CV class defined inline')

In [ ]:
# Load data
train = pd.read_parquet(f'{WORK_DIR}/train_featured.parquet')
val = pd.read_parquet(f'{WORK_DIR}/val_featured.parquet')

# Auto-detect columns
TARGET_COLS = [c for c in train.columns if any(k in c.lower() for k in ['alkalinity', 'conductance', 'phosphorus'])]
STATION_COL = [c for c in train.columns if 'station' in c.lower() or 'gems' in c.lower()][0]
LAT_COL = [c for c in train.columns if 'lat' in c.lower()][0]
LON_COL = [c for c in train.columns if 'lon' in c.lower()][0]
META_COLS = [STATION_COL, LAT_COL, LON_COL] + [c for c in train.columns if 'date' in c.lower() or 'river' in c.lower()]

print(f'Targets: {TARGET_COLS}')
print(f'Station col: {STATION_COL}')
print(f'Train: {train.shape}')

In [ ]:
# Setup spatial CV
cv = LeaveStationGroupOut(n_splits=10, station_col=STATION_COL, random_state=SEED)
fold_map = cv.get_fold_station_mapping(train)

## 📊 FIGURE 1: Spatial CV Fold Distribution Map

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
cmap = plt.cm.get_cmap('tab10', 10)

for fold_idx, stations in fold_map.items():
    fold_data = train[train[STATION_COL].isin(stations)]
    locs = fold_data.groupby(STATION_COL).agg({LAT_COL: 'first', LON_COL: 'first'}).reset_index()
    ax.scatter(locs[LON_COL], locs[LAT_COL], c=[cmap(fold_idx)]*len(locs),
              s=80, alpha=0.8, edgecolors='white', linewidths=0.5,
              label=f'Fold {fold_idx} ({len(stations)} stations, {len(fold_data)} samples)')

ax.set_xlim(16, 33)
ax.set_ylim(-35, -22)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('🗺️ Spatial CV Fold Assignment\n'
             'Each fold holds out entire stations — simulates predicting unseen locations',
             fontsize=13)
ax.legend(fontsize=8, loc='lower left', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_04_cv_folds_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_04_cv_folds_map.png')

## Exp 0: Naïve Baselines

In [ ]:
print('=' * 60)
print('EXP 0: NAÏVE BASELINES')
print('=' * 60)

exp0_results = {}
for target in TARGET_COLS:
    short = target.split('(')[0].strip() if '(' in target else target[:15]
    y = train[target].dropna()
    
    r2_mean = r2_score(y, np.full(len(y), y.mean()))
    r2_median = r2_score(y, np.full(len(y), y.median()))
    
    exp0_results[short] = {'Mean': r2_mean, 'Median': r2_median}
    print(f'  {short}: Mean R²={r2_mean:.4f}, Median R²={r2_median:.4f}')

exp0_mean = np.mean([max(v.values()) for v in exp0_results.values()])
print(f'\nExp 0 Best Mean R² = {exp0_mean:.4f}')

## Exp 1: Default Models with EY Features

In [ ]:
print('=' * 60)
print('EXP 1: EY CORE FEATURES (Landsat + TerraClimate)')
print('=' * 60)

# Identify EY-only features (exclude external data features)
external_keywords = ['soil_', 'elevation', 'precip_sum', 'precip_max', 'temp_max', 'temp_min',
                     'temp_range', 'wind_', 'mine', 'wastewater', 'farmland', 'road', 'osm',
                     'basin_', 'river_', 'population', 'landcover', 'runoff', 'agri_',
                     'dilution', 'weathering', 'clay_organic', 'urban_stress',
                     'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'season_', '_log']

all_numeric = train.select_dtypes(include=[np.number]).columns.tolist()
ey_feature_cols = [c for c in all_numeric 
                   if c not in TARGET_COLS + META_COLS + ['month', 'quarter', 'day_of_year', 'year']
                   and not any(k in c.lower() for k in external_keywords)]

print(f'EY-only features: {len(ey_feature_cols)}')

models_to_test = {
    'Ridge': Ridge(alpha=1.0),
    'RF': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                 random_state=SEED, n_jobs=-1, verbosity=0),
}

In [ ]:
# Run Exp 1 with per-fold tracking
exp1_results = {}      # {target: {model: mean_r2}}
exp1_fold_scores = {}  # {target: {model: [fold_scores]}}

for target in TARGET_COLS:
    short = target.split('(')[0].strip() if '(' in target else target[:15]
    print(f'\n{"="*40}')
    print(f'Target: {short}')
    print(f'{"="*40}')
    
    valid_mask = train[target].notna()
    X = train.loc[valid_mask].reset_index(drop=True)
    y = train.loc[valid_mask, target].reset_index(drop=True)
    
    exp1_results[short] = {}
    exp1_fold_scores[short] = {}
    
    for model_name, model in models_to_test.items():
        fold_scores = []
        for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X)):
            m = clone(model)
            X_tr = X.iloc[train_idx][ey_feature_cols].fillna(0)
            X_te = X.iloc[test_idx][ey_feature_cols].fillna(0)
            m.fit(X_tr, y.iloc[train_idx])
            preds = m.predict(X_te)
            score = r2_score(y.iloc[test_idx], preds)
            fold_scores.append(score)
        
        mean_r2 = np.mean(fold_scores)
        std_r2 = np.std(fold_scores)
        exp1_results[short][model_name] = mean_r2
        exp1_fold_scores[short][model_name] = fold_scores
        print(f'  {model_name}: R² = {mean_r2:.4f} ± {std_r2:.4f}')

# Overall summary
print(f'\n\n{"="*60}')
print('EXP 1 SUMMARY')
print(f'{"="*60}')
summary_df = pd.DataFrame(exp1_results).T
summary_df['Best Model'] = summary_df.idxmax(axis=1)
summary_df.loc['MEAN'] = summary_df.select_dtypes(include=[np.number]).mean()
display(summary_df)

## 📊 FIGURE 2: Exp 0 vs Exp 1 — R² Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

target_names = list(exp1_results.keys())
n_targets = len(target_names)
n_models = len(models_to_test) + 1  # +1 for naive baseline
x = np.arange(n_targets)
width = 0.8 / n_models

colors = {'Naive (mean)': '#9E9E9E', 'Ridge': '#2196F3', 'RF': '#4CAF50', 'XGBoost': '#FF9800'}

# Exp 0 bars
naive_scores = [max(exp0_results[t].values()) for t in target_names]
ax.bar(x - width * (n_models-1)/2, naive_scores, width, label='Exp 0: Naive (mean)',
       color=colors['Naive (mean)'], edgecolor='white', linewidth=0.5)

# Exp 1 bars
for i, model_name in enumerate(models_to_test.keys()):
    scores = [exp1_results[t].get(model_name, 0) for t in target_names]
    offset = x - width * (n_models-1)/2 + width * (i+1)
    ax.bar(offset, scores, width, label=f'Exp 1: {model_name}',
           color=colors.get(model_name, f'C{i+1}'), edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(target_names, fontsize=11)
ax.set_ylabel('R² Score (Spatial CV)', fontsize=12)
ax.set_title('📊 Experiment Comparison: Exp 0 (Naive) vs Exp 1 (EY Features)\n'
             'Higher is better | Spatial CV prevents over-optimism', fontsize=13)
ax.legend(fontsize=10, loc='upper right')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)

# Add R² values on top of bars
for bar_container in ax.containers:
    ax.bar_label(bar_container, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_04_exp0_vs_exp1.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_04_exp0_vs_exp1.png')

## 📊 FIGURE 3: Per-Fold R² Boxplots (Stability Check)

In [ ]:
fig, axes = plt.subplots(1, n_targets, figsize=(6*n_targets, 6))
if n_targets == 1:
    axes = [axes]

for i, target_name in enumerate(target_names):
    ax = axes[i]
    fold_data = exp1_fold_scores.get(target_name, {})
    
    if fold_data:
        data_to_plot = [fold_data[m] for m in models_to_test.keys()]
        bp = ax.boxplot(data_to_plot, labels=list(models_to_test.keys()), patch_artist=True)
        
        box_colors = [colors.get(m, 'C0') for m in models_to_test.keys()]
        for patch, color in zip(bp['boxes'], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    
    ax.set_title(f'{target_name}', fontsize=12)
    ax.set_ylabel('R² per fold')
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='R²=0 (worse than mean)')
    ax.legend(fontsize=8)

fig.suptitle('📦 Per-Fold R² Stability (Exp 1)\n'
             'Wider box = less stable across spatial folds', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_04_fold_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_04_fold_boxplots.png')

## 📊 FIGURE 4: Model × Target Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

heatmap_df = pd.DataFrame(exp1_results)
sns.heatmap(heatmap_df, annot=True, cmap='RdYlGn', center=0, fmt='.4f',
            linewidths=2, square=True, ax=ax, cbar_kws={'label': 'R² Score'})

ax.set_title('🔥 Model × Target R² Heatmap (Exp 1)\n'
             'Green = good, Red = bad', fontsize=13)
ax.set_xlabel('Target Variable')
ax.set_ylabel('Model')

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_04_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_04_heatmap.png')

## Leakage Verification

In [ ]:
forbidden = ['latitude', 'longitude', 'lat', 'lon']
leaks = [c for c in ey_feature_cols if c.lower() in forbidden]

if leaks:
    print(f'⚠️ LEAKAGE DETECTED: {leaks}')
else:
    print('✅ No coordinate leakage detected in features')

## DEVLOG Entry

In [ ]:
print('\n' + '=' * 60)
print('📝 DEVLOG ENTRY — Exp 0 + Exp 1')
print('=' * 60)
print(f'\nExp 0: Naïve baseline → Mean R² = {exp0_mean:.4f}')
print(f'Exp 1: EY Core features ({len(ey_feature_cols)} features)')
for model_name in models_to_test.keys():
    scores = [exp1_results[t].get(model_name, 0) for t in target_names if t != 'MEAN']
    print(f'  {model_name}: Mean R² = {np.mean(scores):.4f}')
print(f'\nSpatial CV: LeaveStationGroupOut, 10 folds, seed={SEED}')
print(f'Leakage: NONE')

print(f'\n📊 Figures saved:')
print(f'  1. fig_04_cv_folds_map.png — Spatial CV fold assignment')
print(f'  2. fig_04_exp0_vs_exp1.png — Baseline comparison bar chart')
print(f'  3. fig_04_fold_boxplots.png — Per-fold stability')
print(f'  4. fig_04_heatmap.png — Model × Target R² heatmap')